# Time-series regression for phase_1

**Date:** 2026-04-18.
**Purpose:** An alternative to per-critic KDE — predict `actual_phase_1` directly from target's observed features. Simpler architecture; no base_rate × KDE × exclusion chain. See if this simpler approach produces more accurate (or at least differently-biased) predictions for h/m targets.

**Setup:** midnight+noon convention (snap = midnight UTC on close-3, day-level timestamps shifted to noon UTC).

**Features (all observable at snap time):**
1. `observed_count` — reviews before snap
2. `first_review_dbc` — first review's days-before-close (proxy for embargo-lift timing)
3. `target_gap` — cohort-computed gap
4. `observed_rate` — count / observed_window_days
5. `rate_last_day` — reviews in final day of observed window
6. `rate_first_day` — reviews in first day post-first-review
7. `top_critic_frac` — fraction of observed reviews from top critics
8. `pub_diversity` — unique publications in observed set
9. `pub_entropy` — Shannon entropy of publication mix
10. `low_activity_frac` — fraction of observed critics with <5 cohort reviews

**Target:** `actual_phase_1` (reviews arriving `[midnight close-3, midnight close_day)`).

**Evaluation:**
- Holdout: 5 h/m movies (deployment-representative)
- Cohort: 5-fold CV on remaining day-level cohort
- Baseline: weighted-KDE prediction (current ship candidate)

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold

from _helpers import (
    reviews, close_date_map, gap_lookup, first_review_ts, gap_for_slug,
    combined_score_with_scores,
    build_weighted_critic_profiles, build_weighted_kde_lambda_model,
    predict_window_custom,
    critic_activity_counts, observed_review_stats,
    CACHE_DIR,
)

SNAP_DAYS = 3

# Use noon-shifted reviews (midnight+noon convention)
reviews_noon = reviews.copy()
day_mask = reviews_noon['timestamp_confidence'] == 'd'
reviews_noon.loc[day_mask, 'estimated_timestamp'] = (
    reviews_noon.loc[day_mask, 'estimated_timestamp'] + pd.Timedelta(hours=12)
)

first_review_ts_noon = (reviews_noon[reviews_noon['movie_slug'].isin(close_date_map)]
                         .groupby('movie_slug')['estimated_timestamp'].min())

activity = critic_activity_counts()

HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']

CACHE = CACHE_DIR / 'time_series_regression.pkl'
print('Ready.')

## Feature extraction + target per movie

In [ ]:
def extract_features_and_target(slug):
    target_close = close_date_map[slug]
    snap_time = target_close.floor('D') - pd.Timedelta(days=SNAP_DAYS)
    snap_dbc_eff = (target_close - snap_time).total_seconds() / 86400
    close_midnight = target_close.floor('D')

    mr_all = reviews_noon[reviews_noon['movie_slug'] == slug]
    obs = mr_all[(mr_all['estimated_timestamp'] < snap_time)
                  & (mr_all['estimated_timestamp'] < target_close)]
    if len(obs) < 3:
        return None

    first_review_ts_target = obs['estimated_timestamp'].min()
    first_review_dbc = (target_close - first_review_ts_target).total_seconds() / 86400
    obs_window_days = first_review_dbc - snap_dbc_eff
    if obs_window_days <= 0:
        return None

    # Target gap from cohort
    target_gap = gap_for_slug(slug)
    if target_gap is None:
        return None

    # Observed stats (using observed_review_stats on a window)
    # Window: from first_review_ts_target to snap_time, duration = obs_window_days
    stats = observed_review_stats(slug, first_review_ts_target, obs_window_days, activity)

    # Rate in final day of observed window (last day before snap)
    last_day_start = snap_time - pd.Timedelta(days=1)
    rate_last_day = ((obs['estimated_timestamp'] >= last_day_start) &
                     (obs['estimated_timestamp'] < snap_time)).sum()

    # Rate in first day post-first-review
    first_day_end = first_review_ts_target + pd.Timedelta(days=1)
    rate_first_day = ((obs['estimated_timestamp'] >= first_review_ts_target) &
                      (obs['estimated_timestamp'] < first_day_end)).sum()

    # Target
    actual_phase1 = int(((mr_all['estimated_timestamp'] >= snap_time) &
                         (mr_all['estimated_timestamp'] < close_midnight)).sum())

    return {
        'target': slug,
        'observed_count': len(obs),
        'first_review_dbc': first_review_dbc,
        'target_gap': target_gap,
        'observed_rate': len(obs) / obs_window_days,
        'rate_last_day': int(rate_last_day),
        'rate_first_day': int(rate_first_day),
        'top_critic_frac': stats['top_critic_frac'],
        'pub_diversity': stats['pub_diversity'],
        'pub_entropy': stats['pub_entropy'],
        'low_activity_frac': stats['low_activity_frac'],
        'actual_phase1': actual_phase1,
    }

rows = []
for slug in close_date_map:
    f = extract_features_and_target(slug)
    if f is not None:
        rows.append(f)

df = pd.DataFrame(rows)
print(f'Extracted features for {len(df)} targets')
print(df[['observed_count', 'first_review_dbc', 'observed_rate', 'rate_last_day',
          'top_critic_frac', 'pub_entropy', 'actual_phase1']].describe().round(2).to_string())

## Baseline: weighted-KDE prediction per target

In [ ]:
def wkde_prediction(slug):
    target_close = close_date_map[slug]
    midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
    snap_time = target_close.floor('D') - pd.Timedelta(days=SNAP_DAYS)
    snap_dbc_eff = (target_close - snap_time).total_seconds() / 86400

    mr_all = reviews_noon[reviews_noon['movie_slug'] == slug]
    obs = mr_all[(mr_all['estimated_timestamp'] < snap_time) & (mr_all['estimated_timestamp'] < target_close)]
    if len(obs) < 3:
        return None
    state = {
        'observed_critics': set(obs['reviewer_name']),
        'observed_count': len(obs),
        'first_review_dbc': float((target_close - obs['estimated_timestamp'].min()).total_seconds() / 86400),
    }
    if state['first_review_dbc'] < snap_dbc_eff + 1.0:
        return None
    target_gap = gap_for_slug(slug)
    if target_gap is None:
        return None
    tw = state['first_review_dbc'] - snap_dbc_eff
    scores = combined_score_with_scores(
        slug, target_gap, state['observed_critics'], tw,
        k=20, alpha=0.5, sigma_gap=8.0,
    )
    if len(scores) < 5:
        return None
    try:
        profiles = build_weighted_critic_profiles(reviews_noon, close_date_map, scores, verbose=False)
        if len(profiles.df) == 0:
            return None
        model = build_weighted_kde_lambda_model(profiles, bandwidth_floor=0.5, bandwidth_ceiling=0.7)
        pred = predict_window_custom(
            model, dbc_from=snap_dbc_eff, dbc_to=midnight_utc_dbc,
            observed_critics=state['observed_critics'],
            observed_count=state['observed_count'],
            first_review_dbc=state['first_review_dbc'],
        )
        return float(pred)
    except Exception:
        return None

df['wkde_pred'] = df['target'].apply(wkde_prediction)
print(f'Computed weighted-KDE baseline for {df["wkde_pred"].notna().sum()} / {len(df)} targets')

## Fit regression models

Train on non-h/m targets (with 5-fold CV within that set), test on h/m holdout.

In [ ]:
FEATURES = [
    'observed_count', 'first_review_dbc', 'target_gap', 'observed_rate',
    'rate_last_day', 'rate_first_day', 'top_critic_frac',
    'pub_diversity', 'pub_entropy', 'low_activity_frac',
]

data = df.dropna(subset=FEATURES + ['actual_phase1']).copy()
print(f'Usable rows: {len(data)}')
print(f'  Day-level cohort: {len(data[~data["target"].isin(HM)])}')
print(f'  H/m holdout:      {len(data[data["target"].isin(HM)])}')

cohort = data[~data['target'].isin(HM)].reset_index(drop=True)
hm = data[data['target'].isin(HM)].reset_index(drop=True)

X_cohort = cohort[FEATURES].values
y_cohort = cohort['actual_phase1'].values
X_hm = hm[FEATURES].values
y_hm = hm['actual_phase1'].values

def eval_model(model, name):
    # Train on cohort, predict hm
    model.fit(X_cohort, y_cohort)
    y_hm_pred = model.predict(X_hm)

    # Also 5-fold CV on cohort for cohort-wide MAE
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cohort_preds = np.zeros(len(cohort))
    for train_idx, test_idx in kf.split(X_cohort):
        m = type(model)(**model.get_params()) if hasattr(model, 'get_params') else type(model)()
        m.fit(X_cohort[train_idx], y_cohort[train_idx])
        cohort_preds[test_idx] = m.predict(X_cohort[test_idx])
    cohort_mae = np.abs(cohort_preds - y_cohort).mean()
    hm_mae = np.abs(y_hm_pred - y_hm).mean()
    hm_me = (y_hm_pred - y_hm).mean()
    return name, cohort_mae, hm_mae, hm_me, y_hm_pred

results = []
print(f'\n{"model":30s}  {"cohort_MAE_CV":>14s}  {"h/m_MAE":>9s}  {"h/m_mean_err":>12s}')
for m, nm in [
    (LinearRegression(), 'OLS'),
    (Ridge(alpha=1.0), 'Ridge(alpha=1)'),
    (Ridge(alpha=10.0), 'Ridge(alpha=10)'),
    (GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42), 'GBM(100, depth3)'),
    (GradientBoostingRegressor(n_estimators=300, max_depth=2, random_state=42), 'GBM(300, depth2)'),
]:
    name, cohort_mae, hm_mae, hm_me, y_hm_pred = eval_model(m, nm)
    results.append((name, cohort_mae, hm_mae, hm_me, y_hm_pred))
    print(f'  {name:30s}  {cohort_mae:14.2f}  {hm_mae:9.2f}  {hm_me:+12.2f}')

# Baseline: weighted-KDE h/m MAE
hm['wkde_err'] = hm['wkde_pred'] - hm['actual_phase1']
wkde_hm_mae = hm['wkde_err'].abs().mean()
wkde_hm_me = hm['wkde_err'].mean()
print(f'\n  Baseline weighted-KDE on h/m:  MAE={wkde_hm_mae:.2f}  mean_err={wkde_hm_me:+.2f}')

## Per-target h/m predictions

In [ ]:
print('H/m per-target predictions:\n')
print(f'{"target":32s} {"actual":>7s} {"wKDE":>8s} ' + ' '.join(f'{n[:15]:>16s}' for n, *_ in results))
for i, (_, r) in enumerate(hm.iterrows()):
    line = f'{r["target"]:32s} {r["actual_phase1"]:7.0f} {r["wkde_pred"]:8.2f} '
    for _, _, _, _, preds in results:
        line += f'{preds[i]:16.2f} '
    print(line)

print()
print('Per-target errors (pred − actual):\n')
print(f'{"target":32s} {"wKDE":>10s} ' + ' '.join(f'{n[:15]:>16s}' for n, *_ in results))
for i, (_, r) in enumerate(hm.iterrows()):
    line = f'{r["target"]:32s} {r["wkde_err"]:+10.2f} '
    for _, _, _, _, preds in results:
        line += f'{preds[i] - r["actual_phase1"]:+16.2f} '
    print(line)

## Feature importances (GBM)

In [ ]:
gbm = GradientBoostingRegressor(n_estimators=300, max_depth=2, random_state=42)
gbm.fit(X_cohort, y_cohort)
imp = pd.Series(gbm.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('GBM feature importances:')
print(imp.round(3).to_string())

## Decision

Compare h/m MAE across the models vs weighted-KDE baseline:
- **Regression wins on h/m MAE AND cohort MAE** → time-series is a better architecture; consider as primary.
- **Regression wins on h/m, loses on cohort** → could blend with KDE based on target type.
- **Regression loses on h/m** → not the answer; KDE stays.
- **All models give ~same error** → we're data-limited, not architecture-limited.